# Match h5 Files (ROreg with correct ROinv)

In [13]:
import src.read_write as rw
import src.reconstruction as recon
from pathlib import Path
import numpy as np
import time


# ---- Directories ----
dir_h5 = Path('data/raw')

dir_reconstruction = Path('data/all_reconstructions')
dir_reconstruction.mkdir(parents=True, exist_ok=True)

dir_b0_maps = Path('data/all_b0_maps')
dir_b0_maps.mkdir(parents=True, exist_ok=True)

dir_run_info = Path('data/all_run_info')
dir_run_info.mkdir(parents=True, exist_ok=True)


list_reg = []
list_inv = []

for file in dir_h5.iterdir():

        if "ES000" in file.name and "ROreg" in file.name:
            list_reg.append(file)


for file in list_reg:
    inv_file = file.with_name(file.name.replace("ROreg", "ROinv"))
    if inv_file.exists():
        list_inv.append(inv_file)

print(list_reg)
print(list_inv)


[PosixPath('data/raw/position3_ROreg_ES000.h5'), PosixPath('data/raw/position2_ROreg_ES000.h5'), PosixPath('data/raw/position1_ROreg_ES000.h5')]
[PosixPath('data/raw/position3_ROinv_ES000.h5'), PosixPath('data/raw/position2_ROinv_ES000.h5'), PosixPath('data/raw/position1_ROinv_ES000.h5')]


# Run IROC

In [14]:
import torch
import numpy as np
from scipy import ndimage
import time

list_ssh_orders = [2, 4, 8, 12]


histories_all = []

for i in range(len(list_reg)):

    histories_position = []

    xml_reg, kspace_reg = rw.load_h5(list_reg[i])
    xml_inv, kspace_inv = rw.load_h5(list_inv[i])

    # ---- Get dwell time from XML -----
    dwell_time = rw.GetParam_from_xml("dwell", xml_reg, print_all=False)[0] * 1e-03

    # ---- Get FOV from XML and convert to meters -----
    fov = rw.GetParam_from_xml("fov", xml_reg, print_all=False) * 1e-03

    fov_torch = torch.tensor(fov, dtype=torch.float32)

    # ---- Make time array -----
    time_array = recon.tse_time_array(kspace_reg,dwell_time=dwell_time,symmetric_time=True)

    time_array_torch = torch.tensor(time_array,dtype=torch.float64)

    # ---- Define readout axis ----
    readout_axis = 0

    # ---- Convert kspace to torch tensors ----
    kspace_reg_torch = torch.tensor(kspace_reg,dtype=torch.complex128)

    kspace_inv_torch = torch.tensor(kspace_inv,dtype=torch.complex128)

    name_reg = list_reg[i].stem
    name_inv = list_inv[i].stem
    position = name_reg.split("_")[0]

    for ssh_order in list_ssh_orders:


        print(
            f"\n\n\n Starting Optimization for "
            f"{position} and SSH order {ssh_order}...\n"
        )

        # ---- Run optimization ----
        d_b0_final, coef, history = recon.iroc_multires(
            k_space_reg=kspace_reg_torch,
            k_space_inv=kspace_inv_torch,
            time_array=time_array_torch,
            readout_axis=readout_axis,
            fov=fov_torch,
            res_steps=[(30, 30, 30), (50, 50, 50)],
            include_full_res=True,
            n_iter=[50, 30],
            max_order=ssh_order,
        )

        # ---- Add identifying information ----
        history["position"] = position
        history["ssh_order"] = ssh_order

        # ---- Store history ----
        histories_position.append(history)

        # ---- Print total runtime ----
        print(
            f"Total runtime for {position} and "
            f"SSH order {ssh_order}: "
            f"{history['total_time']:.2f} s"
        )

        # ---- Convert to NumPy Array ----
        d_b0_final_npy = d_b0_final.detach().cpu().numpy() # type: ignore

        # ---- Scale to match kspace ----
        d_b0_final_npy = np.asarray(ndimage.zoom(d_b0_final_npy,kspace_reg.shape / np.array(d_b0_final_npy.shape),order=1))

        # ---- Calculate Final Image ----
        final_cpr_reg = recon.time_segmented_cpr(kspace_reg,time_array,d_b0_final_npy,t_shift=0,readout_axis=readout_axis)


        # ---- EXPORT ----
        rw.write_nii_from_xml(abs(final_cpr_reg),xml_reg,f'{dir_reconstruction}/{name_reg}_IROC_Order{ssh_order:02d}.nii.gz') #type:ignore


        np.save(f'{dir_b0_maps}/b0_map_{position}_IROC_Order{ssh_order:02d}.npy',d_b0_final_npy)

    # Store all SSH orders for this position
    histories_all.append(histories_position)




 Starting Optimization for position3 and SSH order 2...

Full Resolution: (111, 111, 111)

=== Resolution stage 1/3: (30, 30, 30) ===
Starting optimization...
iter   1 | loss 1.489424e+00 | time 1.22 s | total 1.22 s
iter   2 | loss 9.636141e-01 | time 0.55 s | total 1.77 s
iter   3 | loss 9.624196e-01 | time 0.06 s | total 1.83 s
iter   4 | loss 9.624196e-01 | time 0.06 s | total 1.89 s
Converged at iter 4
Stage 1 finished: 4 iterations | 1.91 s

=== Resolution stage 2/3: (50, 50, 50) ===
Starting optimization...
iter   1 | loss 2.824348e-01 | time 8.90 s | total 8.90 s
iter   2 | loss 2.802336e-01 | time 3.26 s | total 12.16 s
iter   3 | loss 2.802079e-01 | time 0.36 s | total 12.52 s
iter   4 | loss 2.802079e-01 | time 0.36 s | total 12.88 s
Converged at iter 4
Stage 2 finished: 4 iterations | 12.89 s

=== Resolution stage 3/3: (111, 111, 111) ===
Starting optimization...
iter   1 | loss 3.897856e-02 | time 18.41 s | total 18.41 s
iter   2 | loss 3.897856e-02 | time 16.38 s | tot

# History

In [16]:
import pandas as pd

rows = []

for position_histories in histories_all:

    for history in position_histories:

        for stage in history["stages"]:

            rows.append({
                "Position": history["position"],
                "SSH_Order": history["ssh_order"],
                "Stage": stage["stage"],
                "Shape": stage["shape"],
                "Iterations": stage["n_iterations"],
                "Stage_Time_s": stage["actual_stage_time"],
                "Total_Time_s": history["total_time"],
                "Total_Iterations": history["total_iterations"],
            })

df_history = pd.DataFrame(rows)

df_history.to_csv(f"{dir_run_info}/IROC_history.csv", index=False)
df_history

,Position,SSH_Order,Stage,Shape,Iterations,Stage_Time_s,Total_Time_s,Total_Iterations
0,position3,2,1,"(30, 30, 30)",4,1.910239,49.629080,10
1,position3,2,2,"(50, 50, 50)",4,12.890379,49.629080,10
2,position3,2,3,"(111, 111, 111)",2,34.828398,49.629080,10
3,position3,4,1,"(30, 30, 30)",6,4.878345,61.584139,13
4,position3,4,2,"(50, 50, 50)",5,22.551725,61.584139,13
5,position3,4,3,"(111, 111, 111)",2,34.153825,61.584139,13
6,position3,8,1,"(30, 30, 30)",11,10.166543,97.062451,22
7,position3,8,2,"(50, 50, 50)",9,50.987145,97.062451,22
8,position3,8,3,"(111, 111, 111)",2,35.908417,97.062451,22
9,position3,12,1,"(30, 30, 30)",50,56.684957,335.688118,82
